# Follow-up Experiments — Interactive Kaggle & VS Code Runner

Run this notebook **in the Kaggle session attached to this notebook in VS Code** (or directly on Kaggle web).

### Your Datasets on Kaggle:
- **Structured3D**: `lokeshmotwani213/structured3d-panoramic-depth`
- **PanoX**: `lokeshmotwani213/panox-panoramic-depth`
- **Project Code**: `lokeshmotwani213/lokesh-code`

In the Kaggle sidebar under **Input > + Add Input**, search for and attach these datasets.

The setup cell will:
1. Use your attached `lokesh-code` (or existing `/kaggle/working/Lokesh`);
2. Automatically detect and extract the attached PanoX and Structured3D datasets;
3. Download model code/weights (`DA360_large.pth`, `dpt_hybrid_384.pt`);
4. Stream experiment execution in real-time to this VS Code window.

In [1]:
import os
import sys
import subprocess
from pathlib import Path

import torch

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

gpu_status = subprocess.run(["nvidia-smi"], check=False)
if not torch.cuda.is_available() or torch.cuda.device_count() < 1:
    raise RuntimeError(
        "This notebook is attached to a CPU kernel. In Kaggle, select "
        "Settings > Accelerator > GPU T4 x2, then restart and run from the top."
    )
if gpu_status.returncode != 0:
    print("Warning: nvidia-smi returned a non-zero exit code.")

Python: 3.13.16
PyTorch: 2.11.0+cu130
CUDA available: True
CUDA device count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4
Thu Nov  5 19:28:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                           

In [2]:
import os
import sys
import shutil
import subprocess
import zipfile
from pathlib import Path
from typing import Optional

import torch

# ---------------------------------------------------------------------
# A. Select the experiment BEFORE running this setup cell.
# ---------------------------------------------------------------------
RUN = "all"                 # "01", "02", "03", "04", "05", or "all"
LIMIT = None               # Pilot size for 01/04/05; set None for full runs
RUN_FINETUNE_IN_ALL = True  # If True, run 03 finetuning in the "all" run; otherwise skip it

# Leave dataset roots as None for automatic Kaggle Input discovery.
DATA_PATHS = {
    "PANOX_ROOT": None,
    "STRUCTURED3D_ROOT": None,
    "DA360_REPO": "/kaggle/working/DA360",
    "DA360_WEIGHTS": "/kaggle/working/da360_weights/DA360_large.pth",
    "MIDAS_WEIGHTS": "/kaggle/working/midas_weights/dpt_hybrid_384.pt",
}

# Optional GitHub PAT if cloning private repo:
# Optional GitHub PAT: if provided, Kaggle will automatically commit & push results to GitHub after each stage completes!
GITHUB_PAT = os.environ.get("GITHUB_PAT", "")
REPO_URL = (
    "https://github.com/AurindumBanerjee/"
    "Distortion-Aware-Panoramic-Depth-Estimation.git"
)
WORKING = Path("/kaggle/working")
INPUT = Path("/kaggle/input")

if RUN not in {"01", "02", "03", "04", "05", "all"}:
    raise ValueError(f"Unknown RUN={RUN!r}")
if not WORKING.is_dir():
    raise RuntimeError("This does not look like a Kaggle runtime: /kaggle/working is missing.")


# ---------------------------------------------------------------------
# Helper: sanitize all shell scripts and python files
# ---------------------------------------------------------------------
def _sanitize_scripts(root: Path) -> int:
    """Ensure Linux LF line endings, fix escaped bash variable syntax, and chmod +x."""
    fixed = 0
    for sh in root.rglob("*.sh"):
        try:
            raw = sh.read_bytes()
            clean = (
                raw.replace(b"\r\n", b"\n")
                .replace(b"\r", b"\n")
                .replace(b"\\${", b"${")
            )
            if clean != raw:
                sh.write_bytes(clean)
                fixed += 1
            sh.chmod(0o755)
        except Exception:
            pass
    for py in root.rglob("*.py"):
        try:
            raw = py.read_bytes()
            clean = raw.replace(b"\r\n", b"\n").replace(b"\r", b"\n")
            if clean != raw:
                py.write_bytes(clean)
                fixed += 1
        except Exception:
            pass
    return fixed


# ---------------------------------------------------------------------
# B. Locate or bootstrap the project into writable /kaggle/working.
# ---------------------------------------------------------------------
def _is_lokesh_root(path: Path) -> bool:
    return (
        path.is_dir()
        and (path / "run_experiment.py").is_file()
        and (path / "followup_experiments" / "run_all.sh").is_file()
    )


def _is_writable_directory(path: Path) -> bool:
    try:
        probe = path / ".kaggle_write_probe"
        probe.write_text("ok", encoding="utf-8")
        probe.unlink()
        return True
    except OSError:
        return False


def _git_output(args, cwd=None):
    return subprocess.run(
        args, cwd=cwd, text=True, stdout=subprocess.PIPE,
        stderr=subprocess.PIPE, check=False
    )


def _verify_or_update_experiment_revision(lokesh_root: Path) -> None:
    top = _git_output(
        ["git", "-C", str(lokesh_root), "rev-parse", "--show-toplevel"]
    )
    if top.returncode != 0:
        print("Project source has no Git metadata; using files directly from workspace.")
        return
    print("Project Git repository found at:", top.stdout.strip())


candidate_roots = []
if os.environ.get("LOKESH_ROOT"):
    candidate_roots.append(Path(os.environ["LOKESH_ROOT"]).expanduser())

# Common writable and working locations
candidate_roots.extend([
    Path.cwd(),
    Path.cwd() / "Lokesh",
    WORKING / "Lokesh",
    WORKING / "Distortion-Aware-Panoramic-Depth-Estimation" / "Lokesh",
])

# Deep recursive search across all /kaggle/input mounts
if INPUT.is_dir():
    print("Searching for Lokesh code under /kaggle/input...")
    # 1. Look for run_experiment.py anywhere in /kaggle/input
    for run_file in INPUT.rglob("run_experiment.py"):
        folder = run_file.parent
        if _is_lokesh_root(folder):
            print(f"Found code source in input: {folder}")
            candidate_roots.append(folder)

    # 2. Check for attached code archives (.zip)
    for z in INPUT.rglob("*.zip"):
        if any(k in z.name.lower() for k in ("code", "lokesh", "panoramic")):
            dest = WORKING / "Lokesh"
            if not dest.exists():
                print(f"Unpacking attached code archive {z} to {dest}...")
                with zipfile.ZipFile(z) as zf:
                    zf.extractall(WORKING)
            if _is_lokesh_root(dest):
                candidate_roots.append(dest)

# 1. First check if a writable Lokesh root already exists
LOKESH_ROOT = next(
    (p.resolve() for p in candidate_roots if _is_lokesh_root(p) and _is_writable_directory(p)),
    None,
)

# 2. If only found in read-only /kaggle/input, copy into writable /kaggle/working/Lokesh
input_source = next(
    (
        p.resolve() for p in candidate_roots
        if _is_lokesh_root(p)
        and p.resolve().is_relative_to(INPUT.resolve())
    ),
    None,
)

if LOKESH_ROOT is None:
    if input_source is not None:
        destination = WORKING / "Lokesh"
        if destination.exists() and not _is_lokesh_root(destination):
            shutil.rmtree(destination)
        if not destination.exists():
            def _ignore_non_code(_directory, names):
                return {
                    name for name in names
                    if name in {".git", "__pycache__", ".pytest_cache", "outputs", "logs", "weights", "data"}
                }
            print(f"Copying project from read-only mount {input_source} to {destination}...")
            shutil.copytree(input_source, destination, ignore=_ignore_non_code)
        LOKESH_ROOT = destination.resolve()

# 3. If still not found, try git clone
if LOKESH_ROOT is None:
    repo_dir = WORKING / "Distortion-Aware-Panoramic-Depth-Estimation"
    if not repo_dir.exists():
        print("No project found in /kaggle/input. Trying git clone...")
        clone_url = REPO_URL
        if GITHUB_PAT:
            clone_url = REPO_URL.replace("https://", f"https://{GITHUB_PAT}@")
        try:
            subprocess.check_call([
                "git", "clone", "--branch", "main", "--depth", "1", clone_url, str(repo_dir)
            ])
        except Exception as e:
            print("Git clone failed:", e)
    candidate = repo_dir / "Lokesh"
    if _is_lokesh_root(candidate):
        LOKESH_ROOT = candidate.resolve()

if LOKESH_ROOT is None:
    raise RuntimeError(
        "Could not find Lokesh project folder. "
        "Visible in /kaggle/input: " + str([str(p.relative_to(INPUT)) for p in INPUT.rglob('*') if p.is_dir()][:10])
    )

# Sanitize scripts and python files
sanitized = _sanitize_scripts(LOKESH_ROOT)
if sanitized > 0:
    print(f"Sanitized line endings / permissions on {sanitized} files in {LOKESH_ROOT}")

_verify_or_update_experiment_revision(LOKESH_ROOT)
os.environ["LOKESH_ROOT"] = str(LOKESH_ROOT)
os.chdir(LOKESH_ROOT)
if str(LOKESH_ROOT) not in sys.path:
    sys.path.insert(0, str(LOKESH_ROOT))
shared_dir = LOKESH_ROOT / "followup_experiments" / "shared"
if shared_dir.is_dir() and str(shared_dir) not in sys.path:
    sys.path.insert(0, str(shared_dir))
print("Using writable Lokesh project:", LOKESH_ROOT)

# ---------------------------------------------------------------------
# C. Check GPUs and install helper packages.
# ---------------------------------------------------------------------
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Select GPU T4 x2 in the Kaggle session settings.")
gpu_count = torch.cuda.device_count()
print("CUDA devices:", gpu_count)
for i in range(gpu_count):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
if (RUN == "03" or (RUN == "all" and RUN_FINETUNE_IN_ALL)) and gpu_count < 2:
    raise RuntimeError("Experiment 03 requires both T4 GPUs (GPU T4 x2).")
subprocess.run(["nvidia-smi"], check=False)

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "timm==0.6.13", "einops", "huggingface_hub", "OpenEXR>=3.2"
])

# ---------------------------------------------------------------------
# D. Discover attached datasets or extract an attached ZIP into /kaggle/working.
# ---------------------------------------------------------------------
def _classify_test_file(test_file: Path) -> Optional[str]:
    path_text = str(test_file).lower()
    if "panox" in path_text:
        return "PANOX"
    if "structured3d" in path_text or "structured_3d" in path_text:
        return "STRUCTURED3D"
    try:
        sample = test_file.read_text(encoding="utf-8", errors="ignore")[:1_000_000].lower()
    except OSError:
        return None
    if ".exr" in sample:
        return "PANOX"
    if "scene_" in sample:
        return "STRUCTURED3D"
    return None


def _classify_zip(archive: Path) -> Optional[str]:
    path_text = str(archive).lower()
    if "panox" in path_text:
        return "PANOX"
    if "structured3d" in path_text or "structured_3d" in path_text:
        return "STRUCTURED3D"
    try:
        with zipfile.ZipFile(archive) as zf:
            names = zf.namelist()
            if any(name.lower().endswith(".exr") for name in names):
                return "PANOX"
            test_names = [name for name in names if Path(name).name.lower() == "test.txt"]
            for name in test_names[:3]:
                raw = zf.read(name)[:1_000_000].decode("utf-8", errors="ignore").lower()
                if ".exr" in raw:
                    return "PANOX"
                if "scene_" in raw:
                    return "STRUCTURED3D"
    except (OSError, zipfile.BadZipFile, KeyError):
        pass
    return None


def _unique_path(paths, what: str) -> Path:
    unique = sorted({p.resolve() for p in paths})
    if len(unique) == 1:
        return unique[0]
    if not unique:
        raise FileNotFoundError(f"No {what} found.")
    rendered = "\n".join(f"  - {p}" for p in unique)
    raise RuntimeError(
        f"Found multiple possible {what} locations. Set the corresponding DATA_PATHS "
        f"entry to the exact folder or ZIP:\n{rendered}"
    )


def _find_test_root(search_root: Path, dataset_kind: str) -> Optional[Path]:
    if not search_root.exists() or not search_root.is_dir():
        return None
    tests = list(search_root.rglob("test.txt"))
    matching = [p.parent for p in tests if _classify_test_file(p) == dataset_kind]
    if not matching and (search_root / "test.txt").is_file():
        matching = [search_root]
    if len(set(matching)) == 1:
        return matching[0].resolve()
    if len(set(matching)) > 1:
        return _unique_path(matching, f"{dataset_kind} test.txt roots")
    return None


def _extract_archive(archive: Path, dataset_kind: str) -> Path:
    destination = WORKING / "followup_data" / dataset_kind.lower()
    existing = _find_test_root(destination, dataset_kind)
    if existing is not None:
        print("Reusing extracted dataset:", existing)
        return existing

    with zipfile.ZipFile(archive) as zf:
        total_uncompressed = sum(info.file_size for info in zf.infolist())
        free_bytes = shutil.disk_usage(WORKING).free
        if total_uncompressed > free_bytes * 0.90:
            raise RuntimeError(
                f"{archive.name} needs about {total_uncompressed / (1024**3):.1f} GB "
                f"when extracted, but only {free_bytes / (1024**3):.1f} GB is free in "
                "/kaggle/working. Attach the extracted dataset or free working storage."
            )
        destination.mkdir(parents=True, exist_ok=True)
        print(
            f"Extracting {archive.name} ({total_uncompressed / (1024**3):.1f} GB uncompressed) "
            f"to {destination}..."
        )
        zf.extractall(destination)

    resolved = _find_test_root(destination, dataset_kind)
    if resolved is None:
        raise RuntimeError(
            f"Extracted {archive.name} into {destination}, but could not find a {dataset_kind} "
            "test.txt folder inside it."
        )
    return resolved


def resolve_dataset_root(env_key: str, dataset_kind: str) -> Path:
    configured = DATA_PATHS.get(env_key)
    if configured:
        p = Path(configured).expanduser()
        if not p.is_absolute():
            p = (LOKESH_ROOT / p).resolve()
        if p.is_file() and p.suffix.lower() == ".zip":
            resolved = _extract_archive(p, dataset_kind)
        elif p.is_dir():
            resolved = _find_test_root(p, dataset_kind) or p
        else:
            raise FileNotFoundError(f"Configured {env_key}={p} not found.")
    else:
        # Search anywhere under /kaggle/input
        test_roots = [
            p.parent for p in INPUT.rglob("test.txt")
            if _classify_test_file(p) == dataset_kind
        ]
        archives = [
            p for p in INPUT.rglob("*.zip")
            if _classify_zip(p) == dataset_kind
        ]
        if test_roots:
            resolved = _unique_path(test_roots, f"{dataset_kind} dataset roots")
        elif len(archives) == 1:
            resolved = _extract_archive(archives[0], dataset_kind)
        elif len(archives) > 1:
            raise RuntimeError(
                f"Multiple {dataset_kind} ZIPs found: {archives}. Set {env_key} explicitly."
            )
        else:
            raise FileNotFoundError(
                f"Could not auto-detect {dataset_kind}. Attach its Kaggle Dataset in the Input "
                f"panel, or set DATA_PATHS['{env_key}'] to the folder/ZIP. "
                f"Visible inputs: {[p.name for p in INPUT.iterdir()]}"
            )

    os.environ[env_key] = str(resolved.resolve())
    print(f"{env_key} => {resolved}  [test.txt found]")
    return resolved


needs_da360 = RUN in ("01", "02", "03", "all")
needs_midas = RUN in ("04", "05", "all")
needs_panox = RUN in ("02", "03", "04", "05", "all")
needs_structured = RUN in ("01", "all")

if needs_panox:
    panox_root = resolve_dataset_root("PANOX_ROOT", "PANOX")
if needs_structured:
    structured_root = resolve_dataset_root("STRUCTURED3D_ROOT", "STRUCTURED3D")

# ---------------------------------------------------------------------
# E. Download only the model source/checkpoint required by this run.
# ---------------------------------------------------------------------
if needs_da360:
    da360_repo = Path(DATA_PATHS["DA360_REPO"])
    if not (da360_repo / "checkpoint.py").is_file() or not (da360_repo / "networks").is_dir():
        if da360_repo.exists():
            raise RuntimeError(
                f"{da360_repo} exists but is not the expected official DA360 checkout "
                "(checkpoint.py and networks/ are required). Remove/rename that folder or "
                "set DATA_PATHS['DA360_REPO'] to a clean path."
            )
        da360_repo.parent.mkdir(parents=True, exist_ok=True)
        subprocess.check_call([
            "git", "clone", "--depth", "1",
            "https://github.com/antigravity-tech/DA360.git", str(da360_repo)
        ])

    da360_weight = Path(DATA_PATHS["DA360_WEIGHTS"])
    da360_weight.parent.mkdir(parents=True, exist_ok=True)
    if not da360_weight.is_file() or da360_weight.stat().st_size < 1_000_000:
        from huggingface_hub import hf_hub_download
        print("Downloading the DA360 Large checkpoint (about 1.34 GB)...")
        downloaded = hf_hub_download(
            repo_id="hljiang/DA360",
            filename="DA360_large.pth",
            local_dir=str(da360_weight.parent),
        )
        da360_weight = Path(downloaded)
    os.environ["DA360_REPO"] = str(da360_repo.resolve())
    os.environ["DA360_WEIGHTS"] = str(da360_weight.resolve())
    print("DA360 repo:", da360_repo, "| checkpoint:", da360_weight)

if needs_midas:
    midas_weight = Path(DATA_PATHS["MIDAS_WEIGHTS"])
    midas_weight.parent.mkdir(parents=True, exist_ok=True)
    if not midas_weight.is_file() or midas_weight.stat().st_size < 100_000_000:
        midas_url = "https://github.com/isl-org/MiDaS/releases/download/v3/dpt_hybrid_384.pt"
        print("Downloading MiDaS DPT-Hybrid 384 checkpoint (about 470 MB)...")
        try:
            subprocess.check_call(["wget", "-q", "-O", str(midas_weight), midas_url])
        except Exception:
            import urllib.request
            req = urllib.request.Request(midas_url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req) as resp, open(midas_weight, "wb") as out_f:
                shutil.copyfileobj(resp, out_f)
    os.environ["MIDAS_WEIGHTS"] = str(midas_weight.resolve())
    print("MiDaS checkpoint:", midas_weight)

# ---------------------------------------------------------------------
# F. Check EXR support for PanoX and summarize the runtime configuration.
# ---------------------------------------------------------------------
os.environ["OPENCV_IO_ENABLE_OPENEXR"] = "1"
if needs_panox:
    import cv2
    first_line = next(
        (line.split() for line in (panox_root / "test.txt").read_text().splitlines() if line.strip()),
        None,
    )
    if first_line and len(first_line) > 1:
        exr_path = panox_root / first_line[1]
        if exr_path.is_file():
            probe = cv2.imread(
                str(exr_path), cv2.IMREAD_ANYDEPTH | cv2.IMREAD_UNCHANGED
            )
            if probe is None:
                import OpenEXR  # noqa: F401
                print("OpenCV cannot decode the sample EXR; the OpenEXR fallback is available.")
            else:
                print("OpenCV EXR probe succeeded:", exr_path)
        else:
            print("EXR sample path not found from the first list row; loader will validate paths during the run.")

print("\nSetup complete.")
print("Experiment:", RUN, "| pilot LIMIT:", LIMIT)
print("Project root:", LOKESH_ROOT)
print("Outputs will be written under:", LOKESH_ROOT / "followup_experiments")



Searching for Lokesh code under /kaggle/input...
Found code source in input: /kaggle/input/datasets/lokeshmotwani213/lokesh-code/Lokesh
Copying project from read-only mount /kaggle/input/datasets/lokeshmotwani213/lokesh-code/Lokesh to /kaggle/working/Lokesh...
Project source has no Git metadata; using files directly from workspace.
Using writable Lokesh project: /kaggle/working/Lokesh
CUDA devices: 2
GPU 0: Tesla T4
GPU 1: Tesla T4
Thu Nov  5 19:28:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |    

Cloning into '/kaggle/working/DA360'...


DA360_large.pth: reconstructing file:   0%|          |  0.00B / 1.34GB            

DA360_large.pth: downloading bytes:           |  0.00B            

DA360 repo: /kaggle/working/DA360 | checkpoint: /kaggle/working/da360_weights/DA360_large.pth
MiDaS checkpoint: /kaggle/working/midas_weights/dpt_hybrid_384.pt
OpenCV EXR probe succeeded: /kaggle/input/datasets/lokeshmotwani213/panox-panoramic-depth/panox/ClothingStore/ClothingStore/distance/0002.exr

Setup complete.
Experiment: all | pilot LIMIT: None
Project root: /kaggle/working/Lokesh
Outputs will be written under: /kaggle/working/Lokesh/followup_experiments


## Launch the selected experiment

The setup cell above prepares the **same kernel**. Run this launch cell only after setup completes successfully.

Suggested sequence:
- **01:** start with LIMIT=16, then set LIMIT=None for the full Structured3D run.
- **02:** leave all PanoX environments in the list; do not apply a frame limit.
- **04 and 05:** start with LIMIT=2 because these are inference-heavy.
- **03:** trains the optional student and requires both T4 GPUs.
- **all:** runs 01, 02, 04 and 05 sequentially; set RUN_FINETUNE_IN_ALL=True to include 03.

In [3]:
# Run this only after the setup cell has completed successfully.
import os
import sys
import subprocess
from pathlib import Path

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["PYTHON"] = sys.executable
env["CUDA_VISIBLE_DEVICES"] = (
    "0,1" if RUN == "03" or (RUN == "all" and RUN_FINETUNE_IN_ALL) else "0"
)
env["RUN_FINETUNE"] = "1" if RUN == "all" and RUN_FINETUNE_IN_ALL else "0"
if GITHUB_PAT:
    env["GITHUB_PAT"] = GITHUB_PAT

if "LOKESH_ROOT" not in globals() or not LOKESH_ROOT.is_dir():
    raise RuntimeError("LOKESH_ROOT is not set. Run the setup cell first.")

# Sanitize all scripts in LOKESH_ROOT immediately before execution
def _sanitize_scripts(root: Path) -> int:
    fixed = 0
    for sh in root.rglob("*.sh"):
        try:
            raw = sh.read_bytes()
            clean = (
                raw.replace(b"\r\n", b"\n")
                .replace(b"\r", b"\n")
                .replace(b"\\${", b"${")
            )
            if clean != raw:
                sh.write_bytes(clean)
                fixed += 1
            sh.chmod(0o755)
        except Exception:
            pass
    for py in root.rglob("*.py"):
        try:
            raw = py.read_bytes()
            clean = raw.replace(b"\r\n", b"\n").replace(b"\r", b"\n")
            if clean != raw:
                py.write_bytes(clean)
                fixed += 1
        except Exception:
            pass
    return fixed

def _patch_workspace(root: Path) -> int:
    """Fix module shadowing from DA360 and ensure robust dataset loading."""
    patched = 0
    # 1. Patch da360.py
    da360_f = root / "followup_experiments" / "shared" / "da360.py"
    if da360_f.is_file():
        txt = da360_f.read_text(encoding="utf-8")
        if "if repo in sys.path:" not in txt:
            old_finally = "        finally:\n            os.chdir(old)"
            new_finally = (
                "        finally:\n"
                "            os.chdir(old)\n"
                "            if repo in sys.path:\n"
                "                sys.path.remove(repo)\n"
                "            for module_name in list(sys.modules):\n"
                "                if module_name in {'datasets', 'data', 'utils'} or module_name.startswith(('datasets.', 'data.', 'utils.')):\n"
                "                    mod = sys.modules[module_name]\n"
                "                    if getattr(mod, '__file__', None) and str(repo) in str(mod.__file__):\n"
                "                        del sys.modules[module_name]"
            )
            if old_finally in txt:
                da360_f.write_text(txt.replace(old_finally, new_finally), encoding="utf-8", newline="\n")
                patched += 1

    # 2. Patch common.py
    common_f = root / "followup_experiments" / "shared" / "common.py"
    if common_f.is_file():
        txt = common_f.read_text(encoding="utf-8")
        old_load = (
            "def load_dataset(data_cfg: dict, *, height: int | None = None, width: int | None = None,\n"
            "                 limit: int | None = None):\n"
            "    from datasets.panorama import PanoramaDataset"
        )
        new_load = (
            "def load_dataset(data_cfg: dict, *, height: int | None = None, width: int | None = None,\n"
            "                 limit: int | None = None):\n"
            "    if str(ROOT) in sys.path:\n"
            "        sys.path.remove(str(ROOT))\n"
            "    sys.path.insert(0, str(ROOT))\n"
            "    if 'datasets' in sys.modules:\n"
            "        mod = sys.modules['datasets']\n"
            "        if getattr(mod, '__file__', None) and not str(mod.__file__).startswith(str(ROOT)):\n"
            "            for m in list(sys.modules):\n"
            "                if m == 'datasets' or m.startswith('datasets.'):\n"
            "                    del sys.modules[m]\n"
            "    from datasets.panorama import PanoramaDataset"
        )
        if old_load in txt:
            common_f.write_text(txt.replace(old_load, new_load), encoding="utf-8", newline="\n")
            patched += 1

    # 3. Patch 01 run.py
    r1_f = root / "followup_experiments" / "01_scale_only_protocol" / "run.py"
    if r1_f.is_file():
        txt = r1_f.read_text(encoding="utf-8")
        old_order = (
            "    model = load_model(cfg[\"model\"])\n"
            "    if model.spec.space != \"disparity\":\n"
            "        raise SystemExit(f\"Expected a disparity model, got {model.spec}\")\n"
            "    ds = load_dataset(cfg[\"dataset\"], limit=args.limit)"
        )
        new_order = (
            "    ds = load_dataset(cfg[\"dataset\"], limit=args.limit)\n"
            "    model = load_model(cfg[\"model\"])\n"
            "    if model.spec.space != \"disparity\":\n"
            "        raise SystemExit(f\"Expected a disparity model, got {model.spec}\")"
        )
        if old_order in txt:
            r1_f.write_text(txt.replace(old_order, new_order), encoding="utf-8", newline="\n")
            patched += 1

    # 4. Patch 02 run.py
    r2_f = root / "followup_experiments" / "02_domain_conditioned_gate" / "run.py"
    if r2_f.is_file():
        txt = r2_f.read_text(encoding="utf-8")
        old_order = (
            "    model = load_model(cfg[\"model\"])\n"
            "    ds = load_dataset(cfg[\"dataset\"], limit=a.limit)"
        )
        new_order = (
            "    ds = load_dataset(cfg[\"dataset\"], limit=a.limit)\n"
            "    model = load_model(cfg[\"model\"])"
        )
        if old_order in txt:
            txt = txt.replace(old_order, new_order)
            patched += 1
        old_crash = "    if len(envs) < 3:\n        raise RuntimeError(f\"Need at least 3 independent environment groups for LOEO, found: {envs}\")"
        new_crash = (
            "    if len(envs) < 3:\n"
            "        if a.limit:\n"
            "            print(f\"Pilot mode: only {len(envs)} environment group(s) in limited subset: {envs}.\")\n"
            "            print(\"Switching to leave-one-scene-out grouping for pilot run.\")\n"
            "            for sid in groups:\n"
            "                groups[sid] = sid\n"
            "                records[sid][\"environment\"] = sid\n"
            "            envs = sorted(set(groups.values()))\n"
            "        else:\n"
            "            raise RuntimeError(f\"Need at least 3 independent environment groups for LOEO, found: {envs}\")"
        )
        if old_crash in txt:
            txt = txt.replace(old_crash, new_crash)
            patched += 1
        r2_f.write_text(txt, encoding="utf-8", newline="\n")

    return patched

_sanitized = _sanitize_scripts(LOKESH_ROOT)
_patched = _patch_workspace(LOKESH_ROOT)
if _sanitized > 0 or _patched > 0:
    print(f"Prepared workspace: {_sanitized} files sanitized, {_patched} fixes applied.")

if RUN == "all":
    cmd = ["bash", "followup_experiments/run_all.sh"]
    if LIMIT is not None:
        cmd += ["--limit", str(LIMIT)]
else:
    folders = {
        "01": "01_scale_only_protocol",
        "02": "02_domain_conditioned_gate",
        "03": "03_panda_style_finetuning",
        "04": "04_mono360_20view",
        "05": "05_fov_resolution_sweep",
    }
    cmd = ["bash", f"followup_experiments/{folders[RUN]}/run_gpu.sh"]
    if LIMIT is not None and RUN in ("01", "04", "05"):
        cmd += ["--limit", str(LIMIT)]

print("Working directory:", LOKESH_ROOT)
print("Command:", " ".join(cmd))
print("CUDA_VISIBLE_DEVICES:", env["CUDA_VISIBLE_DEVICES"])
print("-" * 70)

# Stream live output line-by-line so VS Code displays progress in real-time
proc = subprocess.Popen(
    cmd,
    cwd=LOKESH_ROOT,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in iter(proc.stdout.readline, ""):
    print(line, end="", flush=True)

proc.stdout.close()
return_code = proc.wait()
if return_code != 0:
    raise subprocess.CalledProcessError(return_code, cmd)
print("" + "=" * 70)
print(f"Experiment {RUN} completed successfully!")

# Auto-sync outputs immediately after run completes
try:
    sync_script = LOKESH_ROOT / "followup_experiments" / "shared" / "sync_stage.py"
    if sync_script.is_file():
        subprocess.run([sys.executable, str(sync_script), RUN], cwd=LOKESH_ROOT)
except Exception as _e:
    print(f"[AUTO-SYNC] Notice: {_e}")



Working directory: /kaggle/working/Lokesh
Command: bash followup_experiments/run_all.sh
CUDA_VISIBLE_DEVICES: 0,1
----------------------------------------------------------------------
Starting 01_scale_only_protocol at Thu Nov  5 07:28:41 PM UTC 2026
xFormers not available
xFormers not available
pretrained model is not available.
Model: {'name': 'da360', 'space': 'disparity', 'scale': 'scale', 'geometry_erp': 'range', 'geometry_perspective': 'range', 'device': 'cuda'}
Dataset: 200 scenes; output: /kaggle/working/Lokesh/followup_experiments/01_scale_only_protocol/outputs/run_20261105_192854
/kaggle/working/DA360/networks/da360.py:74: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.mixed_precision):
[10/200] scene_03304_95556
[20/200] scene_03296_2190
[30/200] scene_03239_31776
[40/200] scene_03245_798000
[50/200] scene_03294_863844
[60/200] scene_03328_553720
[70/200] scene_03240_516

## Student evaluation (after Experiment 03)

After training finishes, copy the printed best.pt path into the cell below. Use an absolute path or a path relative to LOKESH_ROOT. Evaluation uses the held-out environment groups saved with the training run.

In [4]:
# Student evaluation (after Experiment 03)
# Set CHECKPOINT to the path of best.pt (or leave empty to auto-detect the latest best.pt)
CHECKPOINT = ""

if not CHECKPOINT:
    ckpts = sorted( 
        (LOKESH_ROOT / "followup_experiments" / "03_panda_style_finetuning" / "outputs").glob("run_*/best.pt"),
        key=lambda p: p.stat().st_mtime,
    )
    if ckpts:
        CHECKPOINT = str(ckpts[-1])
        print("Auto-detected latest student checkpoint:", CHECKPOINT)

if CHECKPOINT:
    checkpoint_path = Path(CHECKPOINT).expanduser()
    if not checkpoint_path.is_absolute():
        checkpoint_path = (LOKESH_ROOT / checkpoint_path).resolve()
    if not checkpoint_path.is_file():
        raise FileNotFoundError(f"Student checkpoint not found: {checkpoint_path}")

    eval_env = os.environ.copy()
    eval_env["PYTHONUNBUFFERED"] = "1"
    eval_env["CUDA_VISIBLE_DEVICES"] = "0"
    eval_env["PYTHON"] = sys.executable

    cmd = [
        "bash",
        "followup_experiments/03_panda_style_finetuning/evaluate_gpu.sh",
        str(checkpoint_path),
    ]
    print("Evaluating student:", checkpoint_path)
    print("-" * 70)
    proc = subprocess.Popen(
        cmd,
        cwd=LOKESH_ROOT,
        env=eval_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in iter(proc.stdout.readline, ""):
        print(line, end="", flush=True)
    proc.stdout.close()
    rc = proc.wait()
    if rc != 0:
        raise subprocess.CalledProcessError(rc, cmd)
    print("\nStudent evaluation completed successfully!")
    # Auto-sync student evaluation outputs immediately
    try:
        sync_script = LOKESH_ROOT / "followup_experiments" / "shared" / "sync_stage.py"
        if sync_script.is_file():
            subprocess.run([sys.executable, str(sync_script), "03_panda_style_finetuning"], cwd=LOKESH_ROOT)
    except Exception as _e:
        print(f"[AUTO-SYNC] Notice: {_e}")
else:
    print("No student checkpoint found yet. Run Experiment 03 first.")



Auto-detected latest student checkpoint: /kaggle/working/Lokesh/followup_experiments/03_panda_style_finetuning/outputs/run_20261105_205624/best.pt
Evaluating student: /kaggle/working/Lokesh/followup_experiments/03_panda_style_finetuning/outputs/run_20261105_205624/best.pt
----------------------------------------------------------------------
xFormers not available
xFormers not available
pretrained model is not available.
/kaggle/working/DA360/networks/da360.py:74: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.mixed_precision):
[10/50] ClothingStore_0141
[20/50] ClothingStore_0206
[30/50] ModularBakeryShop_0064
[40/50] ModularBakeryShop_0142
[50/50] ModularBakeryShop_0256
Saved /kaggle/working/Lokesh/followup_experiments/03_panda_style_finetuning/outputs/evaluation_20261105_210504/REPORT.md

Student evaluation completed successfully!


## Results and session notes

Every run creates a timestamped folder under:

**Lokesh/followup_experiments/<study>/outputs/run_<timestamp>/**

The main files are REPORT.md, summary.json, per_scene.csv and the run log. They are stored under /kaggle/working because the project checkout is bootstrapped there. Download or save the result folders before ending the Kaggle session.

The setup cell does not start a training/evaluation job automatically: you choose the study, run setup, then execute the launch cell. For Experiment 03, run the student-evaluation cell separately after training.

In [5]:
# Results Dashboard — Comprehensive Inspector for All Experiments
import json
from pathlib import Path
from IPython.display import display, Markdown

experiments = [
    ("01_scale_only_protocol", "Experiment 1: DA360 Scale-Only vs Affine Protocol"),
    ("02_domain_conditioned_gate", "Experiment 2: Domain-Conditioned Rotation Gate"),
    ("03_panda_style_finetuning", "Experiment 3: Student Training & Evaluation"),
    ("04_mono360_20view", "Experiment 4: 20-View 360MonoDepth Comparison"),
    ("05_fov_resolution_sweep", "Experiment 5: FOV x Resolution Sweep"),
]

base_dir = LOKESH_ROOT / "followup_experiments"
print("=" * 70)
print("             FOLLOW-UP EXPERIMENTS RESULTS DASHBOARD                  ")
print("=" * 70 + "\n")

for folder, title in experiments:
    runs = sorted((base_dir / folder / "outputs").glob("run_*"), key=lambda p: p.stat().st_mtime)
    eval_runs = sorted((base_dir / folder / "outputs").glob("evaluation_*"), key=lambda p: p.stat().st_mtime)
    all_runs = runs + eval_runs
    
    if not all_runs:
        print(f"⚪ {title}: [No runs completed yet]")
        continue
        
    latest = all_runs[-1]
    print(f"\n{'=' * 70}")
    print(f"🟢 {title}")
    print(f"   Latest run directory: {latest.name}")
    print(f"{'=' * 70}")
    
    report_file = latest / "REPORT.md"
    if report_file.is_file():
        try:
            display(Markdown(f"## {title}\n\n" + report_file.read_text(encoding="utf-8")))
        except Exception:
            print(report_file.read_text(encoding="utf-8"))
            
    summary_file = latest / "summary.json"
    train_summary = latest / "training_summary.json"
    target_json = summary_file if summary_file.is_file() else (train_summary if train_summary.is_file() else None)
    if target_json:
        try:
            data = json.loads(target_json.read_text(encoding="utf-8"))
            if "best_validation_loss" in data:
                print(f"   Student Training: Best validation loss = {data['best_validation_loss']:.5f}")
            elif "summary" in data:
                print(f"   Summary JSON verified ({len(data['summary'])} protocol entries)")
        except Exception as e:
            print(f"   Could not parse {target_json.name}: {e}")



             FOLLOW-UP EXPERIMENTS RESULTS DASHBOARD                  


🟢 Experiment 1: DA360 Scale-Only vs Affine Protocol
   Latest run directory: run_20261105_192854


## Experiment 1: DA360 Scale-Only vs Affine Protocol

# Experiment 1 — DA360 scale-only evaluation protocol

- Scenes: 200
- Model: da360
- Primary comparison: scale-only disparity fit vs affine disparity fit
- Protocols: align-all, align-equatorial
- Positive bootstrap gain means scale-only has lower AbsRel than affine.

## align-all

| Band | Scale-only median | Affine median | Scale-only vs affine gain (95% CI) |
|---|---:|---:|---:|
| all | 0.0127 | 0.0130 | +0.06% [-0.18%, +1.04%] |
| 45-60 | 0.0100 | 0.0097 | -0.55% [-2.23%, -0.00%] |
| 60-75 | 0.0079 | 0.0076 | -2.10% [-4.98%, -0.14%] |
| 75-90 | 0.0056 | 0.0054 | -0.45% [-6.27%, +1.85%] |

## align-equatorial

| Band | Scale-only median | Affine median | Scale-only vs affine gain (95% CI) |
|---|---:|---:|---:|
| all | 0.0140 | 0.0148 | +2.63% [+0.10%, +5.74%] |
| 45-60 | 0.0112 | 0.0130 | +2.64% [+0.07%, +5.34%] |
| 60-75 | 0.0093 | 0.0112 | +4.51% [+0.02%, +13.14%] |
| 75-90 | 0.0070 | 0.0095 | +5.74% [+0.27%, +17.95%] |

## Interpretation guardrails

DA360 describes its prediction as scale-invariant disparity. The scale-only fit is therefore the
methodologically preferred headline protocol; affine fitting is retained as a sensitivity analysis.
This script does not alter model predictions or fit alignment separately by latitude band.
The same scenes and validity masks are used in paired comparisons.

Raw per-scene metrics are in per_scene.csv; the complete record is summary.json.


   Summary JSON verified (2 protocol entries)

🟢 Experiment 2: Domain-Conditioned Rotation Gate
   Latest run directory: run_20261105_193545


## Experiment 2: Domain-Conditioned Rotation Gate

# Experiment 2 — Leave-one-environment-out domain gate

- Independent PanoX environments: 8
- Candidate thresholds: [30, 40, 45, 50, 60, 70]
- Gate fallback: baseline when best calibration gain < 0.50%

| Strategy | Mean AbsRel | Median AbsRel | Median relative gain |
|---|---:|---:|---:|
| LOEO-selected gate | 0.0389 | 0.0282 | +0.00% |
| Fixed 45° gate | 0.0877 | 0.0621 | -55.44% |
| Per-environment oracle (upper bound) | 0.0720 | 0.0519 | -19.58% |

## Fold decisions

| Held-out environment | Selected threshold | Train environments |
|---|---:|---|
| ClothingStore | baseline | Diner50, FeudalJapan_Meshingun, ModularBakeryShop, Mountain_Environment_Set, SouthOfFrance, Warehouse, YellowParrot |
| Diner50 | baseline | ClothingStore, FeudalJapan_Meshingun, ModularBakeryShop, Mountain_Environment_Set, SouthOfFrance, Warehouse, YellowParrot |
| FeudalJapan_Meshingun | baseline | ClothingStore, Diner50, ModularBakeryShop, Mountain_Environment_Set, SouthOfFrance, Warehouse, YellowParrot |
| ModularBakeryShop | baseline | ClothingStore, Diner50, FeudalJapan_Meshingun, Mountain_Environment_Set, SouthOfFrance, Warehouse, YellowParrot |
| Mountain_Environment_Set | baseline | ClothingStore, Diner50, FeudalJapan_Meshingun, ModularBakeryShop, SouthOfFrance, Warehouse, YellowParrot |
| SouthOfFrance | baseline | ClothingStore, Diner50, FeudalJapan_Meshingun, ModularBakeryShop, Mountain_Environment_Set, Warehouse, YellowParrot |
| Warehouse | baseline | ClothingStore, Diner50, FeudalJapan_Meshingun, ModularBakeryShop, Mountain_Environment_Set, SouthOfFrance, YellowParrot |
| YellowParrot | baseline | ClothingStore, Diner50, FeudalJapan_Meshingun, ModularBakeryShop, Mountain_Environment_Set, SouthOfFrance, Warehouse |

Oracle is diagnostic only and never used as a deployable gate. All frames from one environment
remain in the same fold; no frame-level random split is used.



🟢 Experiment 3: Student Training & Evaluation
   Latest run directory: evaluation_20261105_210504


## Experiment 3: Student Training & Evaluation

# Student/teacher evaluation

GT is used only in this evaluation stage, not by train.py.
Evaluated scenes: 50; held-out groups: ['ClothingStore', 'ModularBakeryShop']

## align-all

| band | DA360 median AbsRel | student median AbsRel |
|---|---:|---:|
| all | 0.0637 | 0.3130 |
| 60-75 | 0.0270 | 0.2837 |
| 75-90 | 0.0130 | 0.3136 |

## align-equatorial

| band | DA360 median AbsRel | student median AbsRel |
|---|---:|---:|
| all | 0.0761 | 0.8123 |
| 60-75 | 0.0435 | 1.1819 |
| 75-90 | 0.0333 | 1.3171 |



🟢 Experiment 4: 20-View 360MonoDepth Comparison
   Latest run directory: run_20261105_195950


## Experiment 4: 20-View 360MonoDepth Comparison

# Experiment 4 — 20-view full-sphere inference

The 20-view system uses icosahedral tangent views, per-view disparity standardisation,
deformable scale/offset alignment over overlapping views, and frustum blending.
It is compared against the baseline and the existing two-pole tangent correction.

## align-all

| method | global median AbsRel | 60–75° median | 75–90° median | calls/image |
|---|---:|---:|---:|---:|
| baseline | 0.1898 | 0.1394 | 0.1514 | 1 |
| two_pole_tangent | 0.1632 | 0.1020 | 0.0950 | 3 |
| mono360_20view | 0.1645 | 0.0801 | 0.0832 | 20 |

## align-equatorial

| method | global median AbsRel | 60–75° median | 75–90° median | calls/image |
|---|---:|---:|---:|---:|
| baseline | 0.2321 | 0.2389 | 0.3468 | 1 |
| two_pole_tangent | 0.1913 | 0.1661 | 0.1659 | 3 |
| mono360_20view | 0.1615 | 0.1429 | 0.1550 | 20 |


   Summary JSON verified (2 protocol entries)

🟢 Experiment 5: FOV x Resolution Sweep
   Latest run directory: run_20261105_202824


## Experiment 5: FOV x Resolution Sweep

# Experiment 5 — FOV and tangent resolution sweep

This probes how view geometry and sampling density affect inference-time polar correction.
It is DAC-motivated, but uses the current MiDaS perspective wrapper; it is not a DAC model reproduction.

## align-all

| config | FOV | size | median AbsRel (75–90°) | polar coverage | calls/image |
|---|---:|---:|---:|---:|---:|
| fov70_size256 | 70° | 256² | 0.1402 | 100.0% | 3 |
| fov70_size384 | 70° | 384² | 0.1404 | 100.0% | 3 |
| fov70_size512 | 70° | 512² | 0.1400 | 100.0% | 3 |
| fov90_size256 | 90° | 256² | 0.1041 | 100.0% | 3 |
| fov90_size384 | 90° | 384² | 0.1058 | 100.0% | 3 |
| fov90_size512 | 90° | 512² | 0.1092 | 100.0% | 3 |
| fov110_size256 | 110° | 256² | 0.0963 | 100.0% | 3 |
| fov110_size384 | 110° | 384² | 0.0950 | 100.0% | 3 |
| fov110_size512 | 110° | 512² | 0.0949 | 100.0% | 3 |
| fov130_size256 | 130° | 256² | 0.0941 | 100.0% | 3 |
| fov130_size384 | 130° | 384² | 0.0919 | 100.0% | 3 |
| fov130_size512 | 130° | 512² | 0.0915 | 100.0% | 3 |

## align-equatorial

| config | FOV | size | median AbsRel (75–90°) | polar coverage | calls/image |
|---|---:|---:|---:|---:|---:|
| fov70_size256 | 70° | 256² | 0.3269 | 100.0% | 3 |
| fov70_size384 | 70° | 384² | 0.3273 | 100.0% | 3 |
| fov70_size512 | 70° | 512² | 0.3269 | 100.0% | 3 |
| fov90_size256 | 90° | 256² | 0.1778 | 100.0% | 3 |
| fov90_size384 | 90° | 384² | 0.1761 | 100.0% | 3 |
| fov90_size512 | 90° | 512² | 0.1807 | 100.0% | 3 |
| fov110_size256 | 110° | 256² | 0.1697 | 100.0% | 3 |
| fov110_size384 | 110° | 384² | 0.1659 | 100.0% | 3 |
| fov110_size512 | 110° | 512² | 0.1644 | 100.0% | 3 |
| fov130_size256 | 130° | 256² | 0.1698 | 100.0% | 3 |
| fov130_size384 | 130° | 384² | 0.1697 | 100.0% | 3 |
| fov130_size512 | 130° | 512² | 0.1702 | 100.0% | 3 |


   Summary JSON verified (2 protocol entries)


## Instant Per-Stage Auto-Sync to Local Repository

**Every experiment now automatically syncs its results immediately upon completion!**
Outputs are streamed to stdout as base64 bundles and saved into this notebook on your local disk in real time.

### Continuous Auto-Sync (Recommended):
In your local terminal, keep this command running while experiments run:
```bash
python Lokesh/scripts/sync_kaggle_outputs.py --watch
```
The moment any stage completes on Kaggle, your local repository will extract its metrics, reports, and logs and create a Git commit automatically.

### One-Shot Sync:
You can also extract completed runs at any time by running:
```bash
python Lokesh/scripts/sync_kaggle_outputs.py
```

### Direct GitHub Push from Kaggle (Optional):
If you set `GITHUB_PAT` in Cell 2, Kaggle will also commit and push results directly to GitHub after each stage!


In [ ]:
# Bundle all experiment outputs and logs for synchronization to local repo
import base64
import io
import zipfile
from pathlib import Path

base_dir = LOKESH_ROOT / "followup_experiments"
zip_buf = io.BytesIO()

included = 0
with zipfile.ZipFile(zip_buf, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in base_dir.rglob("*"):
        if not path.is_file():
            continue
        rel = path.relative_to(base_dir)
        if "outputs" in rel.parts or "logs" in rel.parts:
            if path.suffix.lower() in {".pt", ".pth", ".bin"}:
                continue
            zf.write(path, arcname=str(rel))
            included += 1

zip_bytes = zip_buf.getvalue()
payload = base64.b64encode(zip_bytes).decode("ascii")

# Also save a physical zip in /kaggle/working for direct file browsing
archive_file = WORKING / "all_followup_outputs.zip"
archive_file.write_bytes(zip_bytes)

print(f"Packaged {included} result/log files ({len(zip_bytes) / 1024:.1f} KB).")
print(f"Saved archive to: {archive_file}")
print("=== LOCAL_REPO_SYNC_PAYLOAD_START ===")
print(payload)
print("=== LOCAL_REPO_SYNC_PAYLOAD_END ===")
